# ЛР4 — Нейросетевой анализ тональности (C# / .NET Interactive)

В этой работе (продолжение ЛР3) обучаем CNN‑классификатор тональности для корпуса твитов. Реализация на чистом C# в ноутбуке .NET Interactive, без TorchSharp (для корректной работы на macOS).

Модели:
- Базовый: средний эмбеддинг слов → Linear.
- CNN по словам: сверточные фильтры по n‑граммам слов с глобальным max‑pooling.

Примечание: запрещены LSTM/seq2seq — не используются.


In [ ]:
#r "nuget: CsvHelper, 30.0.1"


The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages CsvHelper, 30.0.1

In [ ]:
using System;
using System.IO;
using System.Linq;
using System.Text;
using System.Globalization;
using System.Text.RegularExpressions;
using System.Collections.Generic;
using CsvHelper;
using CsvHelper.Configuration;


In [ ]:
public class TrainRow { public string textID {get;set;} public string text {get;set;} public string selected_text {get;set;} public string sentiment {get;set;} }
public class TestRow  { public string textID {get;set;} public string text {get;set;} public string sentiment {get;set;} }
public class Example  { public string Text; public int Label; }

static string Normalize(string s) {
    if (string.IsNullOrWhiteSpace(s)) return string.Empty;
    s = s.ToLowerInvariant();
    s = Regex.Replace(s, "https?://\\S+", " " );
    s = Regex.Replace(s, "[\\\"'`]+", "" );
    s = Regex.Replace(s, "[^\\p{L}\\p{Nd}#@ ]+", " ");
    s = Regex.Replace(s, "\\s+", " " ).Trim();
    return s;
}

static string[] Tokenize(string s) => Regex.Split(s, "\\s+").Where(t => t.Length > 0).ToArray();

static List<Example> LoadExamples(string path, bool isTrain) {
    using var reader = new StreamReader(path, Encoding.UTF8);
    var cfg = new CsvConfiguration(CultureInfo.InvariantCulture) { HasHeaderRecord = true, DetectDelimiter = true, IgnoreBlankLines = true, BadDataFound = null };
    using var csv = new CsvReader(reader, cfg);
    var ex = new List<Example>();
    if (isTrain) {
        foreach (var r in csv.GetRecords<TrainRow>()) {
            var t = Normalize(r.text); if (string.IsNullOrWhiteSpace(t) || string.IsNullOrWhiteSpace(r.sentiment)) continue;
            int label = r.sentiment.Trim().ToLowerInvariant() switch { "negative" => 0, "neutral" => 1, "positive" => 2, _ => -1 };
            if (label >= 0) ex.Add(new Example { Text = t, Label = label });
        }
    } else {
        foreach (var r in csv.GetRecords<TestRow>()) {
            var t = Normalize(r.text); if (string.IsNullOrWhiteSpace(t) || string.IsNullOrWhiteSpace(r.sentiment)) continue;
            int label = r.sentiment.Trim().ToLowerInvariant() switch { "negative" => 0, "neutral" => 1, "positive" => 2, _ => -1 };
            if (label >= 0) ex.Add(new Example { Text = t, Label = label });
        }
    }
    return ex;
}

var dataDir = Path.Combine("..", "data");
var trainPath = Path.Combine(dataDir, "train.csv");
var testPath  = Path.Combine(dataDir, "test.csv");
var train = LoadExamples(trainPath, true);
var test  = LoadExamples(testPath,  false);
Console.WriteLine($"Train: {train.Count}, Test: {test.Count}");


Train: 27478, Test: 3533


In [ ]:
static (Dictionary<string,int> word2idx, List<string> idx2word) BuildWordVocab(IEnumerable<string> texts, int maxVocab = 20000, int minFreq = 2) {
    var freq = new Dictionary<string,int>(StringComparer.Ordinal);
    foreach (var t in texts) { foreach (var tok in Tokenize(t)) freq[tok] = freq.GetValueOrDefault(tok) + 1; }
    var items = freq.Where(kv => kv.Value >= minFreq).OrderByDescending(kv => kv.Value).Take(maxVocab - 2).Select(kv => kv.Key).ToList();
    var idx2word = new List<string>(items.Count + 2) { "<pad>", "<unk>" }; idx2word.AddRange(items);
    var word2idx = new Dictionary<string,int>(StringComparer.Ordinal); for (int i = 0; i < idx2word.Count; i++) word2idx[idx2word[i]] = i;
    return (word2idx, idx2word);
}

static int[] ToWordIds(string text, Dictionary<string,int> word2idx, int maxLen) { var pad = 0; var unk = 1; var ids = new int[Math.Max(1, maxLen)]; var toks = Tokenize(text); int len = Math.Min(maxLen, toks.Length); for (int i=0;i<len;i++){ ids[i] = word2idx.TryGetValue(toks[i], out var id)? id: unk; } for (int i=len;i<maxLen;i++) ids[i]=pad; return ids; }

int maxLen = 40; int embDim = 50;
var (word2idx, idx2word) = BuildWordVocab(train.Select(e => e.Text));
Console.WriteLine($"Vocab: {word2idx.Count}");

var rnd = new Random(42);
double[][] Emb(int vocab, int dim) { var e = new double[vocab][]; for (int i=0;i<vocab;i++){ e[i]=new double[dim]; if (i==0) continue; for (int d=0; d<dim; d++) e[i][d] = (rnd.NextDouble()-0.5)*0.2; } return e; }
var E = Emb(word2idx.Count, embDim);

// Закодированные датасеты (обрежем для быстроты демонстрации)
int maxTrain = Math.Min(10000, train.Count); int maxTest = Math.Min(3000, test.Count);
var trainEnc = train.Take(maxTrain).Select(e => (ToWordIds(e.Text, word2idx, maxLen), e.Label)).ToList();
var testEnc  = test .Take(maxTest ).Select(e => (ToWordIds(e.Text, word2idx, maxLen), e.Label)).ToList();
Console.WriteLine($"Encoded: train={trainEnc.Count}, test={testEnc.Count}");


Vocab: 10403


Encoded: train=10000, test=3000


In [ ]:
public class CnnWord
{
    readonly int[] K = new[]{3,4,5};
    readonly int F;
    readonly int D;
    readonly double[][][] W; // [kIdx][f][K*D]
    readonly double[][] B;   // [kIdx][f]
    readonly double[][] W2;  // [C][F*len(K)]
    readonly double[]  B2;   // [C]
    readonly double[][] E;
    readonly int C;
    readonly Random rnd;

    public CnnWord(double[][] emb, int classes=3, int filters=32){ E=emb; D=emb[0].Length; C=classes; F=filters; rnd=new Random(1);
        W = new double[K.Length][][]; B = new double[K.Length][]; for (int ki=0; ki<K.Length; ki++){ int sz=K[ki]*D; W[ki]=new double[F][]; B[ki]=new double[F]; for (int f=0; f<F; f++){ W[ki][f]=new double[sz]; for (int i=0;i<sz;i++) W[ki][f][i]=(rnd.NextDouble()-0.5)*0.2; B[ki][f]=0; } }
        int M=F*K.Length; W2=new double[C][]; for (int c=0;c<C;c++){ W2[c]=new double[M]; for (int i=0;i<M;i++) W2[c][i]=(rnd.NextDouble()-0.5)*0.2; } B2=new double[C];
    }

    static double ReLU(double x)=> x>0?x:0;
    static int ArgMax(double[] a){int m=0; for (int i=1;i<a.Length;i++) if (a[i]>a[m]) m=i; return m;}

    // forward for one sample; returns logits and caches for backprop
    public (double[] logits, (int pos,double pre)[][] cache, double[] feat) Forward(int[] ids){
        int L = ids.Length; var feats = new double[F*K.Length]; var cache = new (int,double)[K.Length][];
        for (int ki=0; ki<K.Length; ki++){ int k=K[ki]; cache[ki]=new (int,double)[F]; for (int f=0; f<F; f++){ double best=-1e30; int bestPos=-1; for (int p=0; p<=L-k; p++){ double s=B[ki][f]; int widx=0; for (int t=0;t<k;t++){ var row=E[ ids[p+t] ]; for (int d=0; d<D; d++){ s += W[ki][f][widx++] * row[d]; } } double pre=s; s=ReLU(s); if (s>best){ best=s; bestPos=p; cache[ki][f]=(bestPos, pre); } } feats[ki*F+f]=best; } }
        var logits = new double[C]; for (int c=0;c<C;c++){ double z=B2[c]; for (int i=0;i<feats.Length;i++) z+=W2[c][i]*feats[i]; logits[c]=z; }
        return (logits, cache, feats);
    }

    static double[] Softmax(double[] z){ double max=z.Max(); double sum=0; var p=new double[z.Length]; for (int i=0;i<z.Length;i++){ p[i]=Math.Exp(z[i]-max); sum+=p[i]; } for (int i=0;i<z.Length;i++) p[i]/=sum; return p; }

    public (double loss, double acc) TrainEpoch(List<(int[] ids,int y)> data, int batch=32, double lr=0.01){
        // grads
        var gW = new double[K.Length][][]; var gB = new double[K.Length][]; for (int ki=0; ki<K.Length; ki++){ gW[ki]=new double[F][]; gB[ki]=new double[F]; for (int f=0; f<F; f++){ gW[ki][f]=new double[K[ki]*D]; } }
        var gW2 = new double[C][]; for (int c=0;c<C;c++){ gW2[c]=new double[F*K.Length]; } var gB2 = new double[C];
        double totLoss=0; int correct=0; int n=0;
        var idx = Enumerable.Range(0, data.Count).ToArray(); for (int i=0;i<idx.Length;i++){ int j=rnd.Next(i,idx.Length); (idx[i],idx[j])=(idx[j],idx[i]); }
        int bsCount=0;
        foreach (var ii in idx){ var (ids,y)=data[ii]; var (logits, cache, feat)=Forward(ids); var p=Softmax(logits); int py=ArgMax(p); if (py==y) correct++;
            double loss = -Math.Log(Math.Max(1e-12, p[y])); totLoss += loss; n++;
            // dL/dz = p; p[y]-=1
            p[y]-=1; // now p is dL/dz
            // top layer grads
            for (int c=0;c<C;c++){ gB2[c]+=p[c]; for (int i2=0;i2<feat.Length;i2++) gW2[c][i2]+=p[c]*feat[i2]; }
            // backprop to pooled feat
            var dfeat = new double[feat.Length]; for (int i2=0;i2<feat.Length;i2++){ double s=0; for (int c=0;c<C;c++) s+=W2[c][i2]*p[c]; dfeat[i2]=s; }
            // conv grads (only at argmax, and only if pre>0)
            for (int ki=0; ki<K.Length; ki++){ int k=K[ki]; for (int f=0; f<F; f++){ double g = dfeat[ki*F+f]; var (pos, pre)=cache[ki][f]; if (pre<=0) continue; gB[ki][f]+=g; int widx=0; for (int t=0;t<k;t++){ var row=E[ ids[pos+t] ]; for (int d=0; d<D; d++){ gW[ki][f][widx++] += g * row[d]; } } } }
            bsCount++; if (bsCount==batch){ // apply update
                double inv=1.0/bsCount; for (int c=0;c<C;c++){ B2[c]-=lr*gB2[c]*inv; for (int i2=0;i2<gW2[c].Length;i2++){ W2[c][i2]-=lr*gW2[c][i2]*inv; gW2[c][i2]=0; } gB2[c]=0; }
                for (int ki=0; ki<K.Length; ki++){ for (int f=0; f<F; f++){ B[ki][f]-=lr*gB[ki][f]*inv; gB[ki][f]=0; for (int w=0; w<gW[ki][f].Length; w++){ W[ki][f][w]-=lr*gW[ki][f][w]*inv; gW[ki][f][w]=0; } } }
                bsCount=0;
            }
        }
        // leftover
        if (bsCount>0){ double inv=1.0/bsCount; for (int c=0;c<C;c++){ B2[c]-=lr*gB2[c]*inv; for (int i2=0;i2<gW2[c].Length;i2++){ W2[c][i2]-=lr*gW2[c][i2]*inv; } } for (int ki=0; ki<K.Length; ki++){ for (int f=0; f<F; f++){ B[ki][f]-=lr*gB[ki][f]*inv; for (int w=0; w<gW[ki][f].Length; w++){ W[ki][f][w]-=lr*gW[ki][f][w]*inv; } } } }
        return (totLoss/Math.Max(1,n), (double)correct/Math.Max(1,n));
    }

    public (double loss, double acc) Evaluate(List<(int[] ids,int y)> data){ double tot=0; int corr=0; int n=0; foreach (var (ids,y) in data){ var (logits,_,_) = Forward(ids); var p=Softmax(logits); int py=ArgMax(p); if (py==y) corr++; tot+= -Math.Log(Math.Max(1e-12, p[y])); n++; } return (tot/Math.Max(1,n),(double)corr/Math.Max(1,n)); }

    public string Predict(string text, Dictionary<string,int> word2idx, int maxLen){ var ids = Lab4Helpers.ToWordIds(text, word2idx, maxLen); var (logits,_,_)=Forward(ids); var p=Softmax(logits); int py=ArgMax(p); string lbl = py==0?"negative": py==1?"neutral":"positive"; return $"{lbl} (p={p[py]:F3})"; }
}

public static class Lab4Helpers { public static int[] ToWordIds(string text, Dictionary<string,int> word2idx, int maxLen){ var pad=0; var unk=1; var ids=new int[Math.Max(1,maxLen)]; var toks=Regex.Split(text, "\\s+"); int len=Math.Min(maxLen, toks.Length); for (int i=0;i<len;i++){ ids[i]=word2idx.TryGetValue(toks[i], out var id)? id: unk; } for (int i=len;i<maxLen;i++) ids[i]=pad; return ids; } }


In [ ]:
var cnn = new CnnWord(E, classes:3, filters:32);
int epochs=2; int batch=32; double lr=0.01;
for (int ep=1; ep<=epochs; ep++){ var (trL, trA)=cnn.TrainEpoch(trainEnc, batch, lr); var (teL, teA)=cnn.Evaluate(testEnc); Console.WriteLine($"[CNN] epoch {ep}: train loss={trL:F4}, acc={trA:F3} | test loss={teL:F4}, acc={teA:F3}"); }


[CNN] epoch 1: train loss=1,0910, acc=0,402 | test loss=1,0873, acc=0,408


[CNN] epoch 2: train loss=1,0881, acc=0,403 | test loss=1,0865, acc=0,408


In [ ]:
// Baseline: avg-emb + linear (train только верхний слой)
var rnd2 = new Random(2);
double[] AvgFeat(int[] ids){ var f=new double[embDim]; int cnt=0; foreach (var id in ids){ if (id==0) continue; var v=E[id]; for (int d=0; d<embDim; d++) f[d]+=v[d]; cnt++; } if (cnt==0) return f; for (int d=0; d<embDim; d++) f[d]/=cnt; return f; }
var Wb = new double[3][]; for (int c=0;c<3;c++){ Wb[c]=new double[embDim]; for (int d=0; d<embDim; d++) Wb[c][d]=(rnd2.NextDouble()-0.5)*0.2; } var Bb = new double[3];
(double loss,double acc) TrainBaselineEpoch(){ double loss=0; int corr=0; int n=0; foreach (var (ids,y) in trainEnc){ var f=AvgFeat(ids); var z=new double[3]; for (int c=0;c<3;c++){ double s=Bb[c]; for (int d=0; d<embDim; d++) s+=Wb[c][d]*f[d]; z[c]=s; } double max=z.Max(); double sum=0; var p=new double[3]; for (int i=0;i<3;i++){ p[i]=Math.Exp(z[i]-max); sum+=p[i]; } for (int i=0;i<3;i++) p[i]/=sum; int py = Array.IndexOf(p, p.Max()); if (py==y) corr++; loss += -Math.Log(Math.Max(1e-12, p[y])); p[y]-=1; double lr0=0.05; Bb[0]-=lr0*p[0]; Bb[1]-=lr0*p[1]; Bb[2]-=lr0*p[2]; for (int c=0;c<3;c++){ for (int d=0; d<embDim; d++) Wb[c][d]-=lr0*p[c]*f[d]; } n++; } return (loss/Math.Max(1,n),(double)corr/Math.Max(1,n)); }
(double loss,double acc) EvalBaseline(List<(int[] ids,int y)> data){ double loss=0; int corr=0; int n=0; foreach (var (ids,y) in data){ var f=AvgFeat(ids); var z=new double[3]; for (int c=0;c<3;c++){ double s=Bb[c]; for (int d=0; d<embDim; d++) s+=Wb[c][d]*f[d]; z[c]=s; } double max=z.Max(); double sum=0; var p=new double[3]; for (int i=0;i<3;i++){ p[i]=Math.Exp(z[i]-max); sum+=p[i]; } for (int i=0;i<3;i++) p[i]/=sum; int py = Array.IndexOf(p, p.Max()); if (py==y) corr++; loss += -Math.Log(Math.Max(1e-12, p[y])); n++; } return (loss/Math.Max(1,n),(double)corr/Math.Max(1,n)); }
for (int ep=1; ep<=2; ep++){ var (trL,trA)=TrainBaselineEpoch(); var (teL,teA)=EvalBaseline(testEnc); Console.WriteLine($"[Baseline] epoch {ep}: train loss={trL:F4}, acc={trA:F3} | test loss={teL:F4}, acc={teA:F3}"); }


[Baseline] epoch 1: train loss=1,0920, acc=0,393 | test loss=1,0789, acc=0,410


[Baseline] epoch 2: train loss=1,0840, acc=0,402 | test loss=1,0727, acc=0,424


In [ ]:
var samples = new[]{
    "Absolutely love the latest update, everything works flawlessly and makes me so happy!",
    "Customer support was terrible: half of my order was missing and nobody apologized.",
    "It's fine I guess, not too bad but nothing exciting either."
};
foreach (var s in samples){ var pred=cnn.Predict(Normalize(s), word2idx, maxLen); Console.WriteLine(s); Console.WriteLine($" -> {pred}"); }


Absolutely love the latest update, everything works flawlessly and makes me so happy!


 -> neutral (p=0,400)


Customer support was terrible: half of my order was missing and nobody apologized.


 -> neutral (p=0,410)


It's fine I guess, not too bad but nothing exciting either.


 -> neutral (p=0,406)
